# 07 — QUBO / Ising

Formulate combinatorial optimisation problems as QUBO / Ising models and solve them.

Covers:
- Max-Cut and Knapsack problem formulation
- Brute-force solver (n ≤ 20) and simulated annealing (any n)
- QUBO ↔ Ising round-trip
- D-Wave Ocean SDK export
- QAOA circuit generation

```bash
pip install quprep
```

In [ ]:
import numpy as np

from quprep.qubo import (
    ising_to_qubo,
    knapsack,
    max_cut,
    qaoa_circuit,
    qubo_to_ising,
    solve_brute,
    solve_sa,
)

## 1. Max-Cut — problem formulation

Find the partition of a graph's vertices that maximises the number of cut edges.

In [ ]:
adj = np.array([[0, 1, 1, 0],
                [1, 0, 1, 1],
                [1, 1, 0, 1],
                [0, 1, 1, 0]], dtype=float)

q = max_cut(adj)
print(f"Variables : {q.Q.shape[0]}")
print(f"Offset    : {q.offset}")

# Evaluate a specific assignment
x_test = np.array([0.0, 1.0, 0.0, 1.0])
print(f"evaluate([0,1,0,1]) = {q.evaluate(x_test):.4f}")

## 2. Brute-force solver (exact, n ≤ 20)

In [ ]:
sol = solve_brute(q)
bits = "".join(str(int(b)) for b in sol.x)
print(f"Exact solution : x={bits}  energy={sol.energy:.4f}")

## 3. QUBO ↔ Ising round-trip

In [ ]:
ising = q.to_ising()
print(f"Ising h : {ising.h}")
print(f"Ising J :\n{ising.J}")

q2 = ising.to_qubo()
print(f"Round-trip Q matches: {np.allclose(q.Q, q2.Q)}")

q3 = ising_to_qubo(qubo_to_ising(q))
print(f"ising_to_qubo() round-trip: {np.allclose(q.Q, q3.Q)}")

## 4. D-Wave Ocean SDK export

In [ ]:
dwave = q.to_dwave()
print(f"D-Wave BQM dict ({len(dwave)} entries):")
for (i, j), v in list(dwave.items())[:5]:
    print(f"  ({i},{j}): {v:.4f}")

## 5. Knapsack with simulated annealing

In [ ]:
weights = np.array([2.0, 3.0, 4.0, 5.0, 6.0])
values  = np.array([3.0, 4.0, 5.0, 6.0, 7.0])
kq = knapsack(weights, values, capacity=10.0)

sol_sa    = solve_sa(kq, n_steps=20_000, restarts=3, seed=42)
sol_exact = solve_brute(kq)

bits_sa    = "".join(str(int(b)) for b in sol_sa.x[:len(weights)])
bits_exact = "".join(str(int(b)) for b in sol_exact.x[:len(weights)])
print(f"SA    : x={bits_sa}  energy={sol_sa.energy:.4f}")
print(f"Exact : x={bits_exact}  energy={sol_exact.energy:.4f}")

## 6. QAOA circuit generation

In [ ]:
small_adj = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0]], dtype=float)
qsmall = max_cut(small_adj)
qasm = qaoa_circuit(qsmall, p=2)

lines = qasm.strip().splitlines()
print(f"QAOA circuit ({len(lines)} lines):")
for line in lines[:8]:
    print(f"  {line}")
print("  ...")